# opts

> Naming a model once, and configuring it once.

This is the module the refactor exists for.

Before it, every backend re-declared the same dozen constructor parameters by hand, and a caller wanting to set a context window had to know it is `n_ctx` on llama, `eng_kw['max_num_tokens']` on litert and `ctx_limit` on a hosted model. Downstream packages ended up carrying a translation table per runtime, which drifted every time a backend changed.

Here there is one name for each thing (`ChatOpts`), one place a backend says what it calls that thing (`_opt_map`), and one function that does the translation (`backend_kw`).

In [ ]:
#| default_exp opts

In [ ]:
#| export
import warnings
from dataclasses import dataclass, field, fields, replace
from importlib import import_module
from fastcore.all import Path, L
from urai.caps import Caps, hosted_caps, hosted_ctx, is_path, DFLT_CTX

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail

## The runtime registry

urai knows nothing about any particular backend. A backend package registers itself, naming its `Chat` subclass and the substrings that identify a model as its own. The name is a dotted path rather than the class, so the registry stays import-free and nothing is loaded until a chat is actually built. A class object works too, for a backend defined in the process rather than installed.

Registration order is inference order, and it matters. `hf.co/…-GGUF` is how Ollama addresses a GGUF repo, so ollama has to be asked before llama or every Ollama model would be read as a local file.

In [ ]:
#| export
@dataclass(frozen=True)
class Runtime:
    "One registered backend: where its `Chat` lives and how to recognise a model of its own."
    name: str
    cls: object                # `'pkg.mod.ChatSubclass'`, or the class itself
    pats: tuple = ()           # id or path substrings that infer this runtime
    optional: bool = False     # ...so a missing import can suggest an extra instead of raising
    extra: str = ''            # the pip extra that installs it; defaults to `name`
    local: bool = True         # does it run on this machine?
    caps: object = None        # `caps(model, model_path)` prober, dotted or callable
    ctx: int = 0               # fixed window, for a runtime whose models all share one

    @property
    def install(self):
        "What to tell someone whose optional backend is missing."
        pkg = self.cls.split('.')[0] if isinstance(self.cls, str) else self.name
        return f"pip install '{pkg}[{self.extra or self.name}]'"

RUNTIMES = {}   #: name -> `Runtime`, in inference order

def register_runtime(rt, dflt=False):
    "Add `rt` to the registry. `dflt=True` also makes it what a bare `Chat()` builds."
    global dflt_runtime
    RUNTIMES[rt.name] = rt
    if dflt: dflt_runtime = rt.name
    return rt

dflt_runtime = None   #: runtime for a `Chat()` with no model named

In [ ]:
#| hide
# a throwaway backend, so this notebook can exercise the registry without importing one
RUNTIMES.clear()
register_runtime(Runtime('ollama', 'demo.ollama.OllamaChat', ('hf.co/',), local=True))
register_runtime(Runtime('llama', 'demo.llama.LlamaChat', ('.gguf',), optional=True, local=True))
register_runtime(Runtime('remote', 'demo.remote.RemoteChat', ('gpt-', 'claude-'), local=False),
                 dflt=True)
test_eq(list(RUNTIMES), ['ollama', 'llama', 'remote'])
test_eq(dflt_runtime, 'remote')
test_eq(RUNTIMES['llama'].install, "pip install 'demo[llama]'")
test_eq(Runtime('local', object).install, "pip install 'local[local]'")

## Naming a model

Three ways to say which runtime, tried in order: an explicit `runtime=`, a `runtime/` prefix on the name, or the shape of the id itself. When none of them answers, the error says all three ways rather than only the one the caller missed.

In [ ]:
#| export
def split_runtime(model):
    "Split `'runtime/model'` into `(runtime, model)`. The prefix must name a registered runtime."
    if isinstance(model, str) and '/' in model:
        b, m = model.split('/', 1)
        if b in RUNTIMES: return b, m
    return None, model

def infer_runtime(model):
    "Guess a runtime from the shape of a model id or path, else `None`."
    s = str(model or '').lower()
    if not s: return None
    return next((n for n, rt in RUNTIMES.items() if any(p in s for p in rt.pats)), None)

def resolve_runtime(model=None, runtime=None, model_path=None):
    "Resolve `(runtime, model)` from an explicit `runtime`, a `runtime/` prefix, or the id shape."
    pre, model = split_runtime(model)
    nm = runtime or pre or infer_runtime(model) or infer_runtime(model_path)
    if nm is None and model is None and model_path is None: nm = dflt_runtime
    if nm is None: raise ValueError(
        f"Can't tell which backend {model!r} needs. Pass runtime={'|'.join(map(repr, RUNTIMES))}, "
        f"prefix the name (e.g. 'llama/{model}'), or give a full repo id or path.")
    if nm not in RUNTIMES: raise ValueError(f'Unknown runtime {nm!r}; known: {", ".join(RUNTIMES)}.')
    return nm, model

In [ ]:
test_eq(split_runtime('llama/qwen3-4b'), ('llama', 'qwen3-4b'))
test_eq(split_runtime('openai/gpt-5.1'), (None, 'openai/gpt-5.1'))   # 'openai' is not a runtime
test_eq(split_runtime('plain'), (None, 'plain'))

In [ ]:
test_eq(infer_runtime('qwen3-4b.gguf'), 'llama')
test_eq(infer_runtime('hf.co/unsloth/qwen3-GGUF'), 'ollama')  # registered first, so it wins
test_eq(infer_runtime('gpt-5.1'), 'remote')
test_eq(infer_runtime('mystery-model'), None)

In [ ]:
test_eq(resolve_runtime('gpt-5.1'), ('remote', 'gpt-5.1'))
test_eq(resolve_runtime('llama/anything'), ('llama', 'anything'))
test_eq(resolve_runtime('anything', runtime='llama'), ('llama', 'anything'))
test_eq(resolve_runtime(), ('remote', None))                   # nothing named: the default
test_eq(resolve_runtime(None, model_path='/m/x.gguf'), ('llama', None))

In [ ]:
test_fail(lambda: resolve_runtime('mystery'), contains="Can't tell which backend")
test_fail(lambda: resolve_runtime('x', runtime='nope'), contains='Unknown runtime')

## A resolved model

`ModelSpec` is one model, decided: which runtime runs it, what that runtime should be handed, how big its window is, and any runtime options that travel with it. It is frozen, so it can be cached, compared and passed around without anyone editing it by accident.

`opts` is excluded from equality because it may carry a resolved API key, and two specs for the same model are the same model.

In [ ]:
#| export
@dataclass(frozen=True)
class ModelSpec:
    "One model, resolved: which runtime runs it, what to call it, and how big it is."
    name: str                  # what the caller typed
    runtime: str
    model_id: str              # what the backend is given
    ctx: int = DFLT_CTX        # context window in tokens
    note: str = ''             # anything worth showing about how this was resolved
    opts: dict = field(default_factory=dict, compare=False)   # runtime options; never a secret

    @property
    def rt(self): return RUNTIMES[self.runtime]
    @property
    def local(self): return self.rt.local
    def __str__(self): return f'{self.name} ({self.model_id})'

def resolve(name=None, runtime=None, model_path=None, ctx=None, **opts):
    "A `ModelSpec` for `name`, working out the runtime and the window."
    rt_name, model_id = resolve_runtime(name, runtime, model_path)
    rt, note = RUNTIMES[rt_name], ''
    if ctx is None:
        if rt.ctx: ctx = rt.ctx
        elif rt.local: ctx = DFLT_CTX
        else: ctx, note = hosted_ctx(model_id)
    return ModelSpec(name or model_id or rt_name, rt_name, model_id, ctx, note, opts)

In [ ]:
s = resolve('gpt-5.1')
test_eq((s.name, s.runtime, s.model_id, s.local), ('gpt-5.1', 'remote', 'gpt-5.1', False))
assert s.ctx > 100_000 and s.note == ''
test_eq(str(s), 'gpt-5.1 (gpt-5.1)')

In [ ]:
s = resolve('llama/qwen3-4b', ctx=8192, n_gpu_layers=99)
test_eq((s.runtime, s.model_id, s.ctx, s.opts), ('llama', 'qwen3-4b', 8192, {'n_gpu_layers': 99}))
test_eq(s.local, True)
test_eq(resolve('unknown-hosted', runtime='remote').note,
        'context window unknown, assuming 128k')

In [ ]:
# `opts` is out of the comparison: the same model twice is the same model
test_eq(resolve('gpt-5.1', temp=0.1), resolve('gpt-5.1', temp=0.9))
test_eq(resolve('gpt-5.1') == resolve('gpt-4o'), False)

## Capabilities

`model_caps` routes to whichever prober the runtime registered, and falls back to the hosted table for a runtime that registered none. This is the seam that lets caps stay ignorant of backends and backends stay ignorant of each other.

In [ ]:
#| export
def load_ref(o):
    "`o` itself, or the object at dotted path `'pkg.mod.name'`."
    if not isinstance(o, str): return o
    mod, _, nm = o.rpartition('.')
    return getattr(import_module(mod), nm)

def model_caps(model=None, runtime=None, model_path=None):
    "What `model` accepts and returns, resolved without loading it."
    try: nm, mid = resolve_runtime(model, runtime, model_path)
    except Exception: nm, mid = runtime, model
    mid = str(mid if mid is not None else (model or ''))
    rt = RUNTIMES.get(nm)
    if rt is None: return hosted_caps(mid)
    if rt.caps:
        try: return load_ref(rt.caps)(mid, model_path) or Caps()
        except Exception: return Caps()
    return Caps() if rt.local else hosted_caps(mid)

In [ ]:
test_eq(model_caps('gpt-5.1').source, 'fastllm')
test_eq(model_caps('qwen3-4b.gguf').source, 'default')     # local, no prober registered
test_eq(model_caps('gpt-5.1', runtime='nope').source, 'fastllm')  # unknown runtime, still asked

In [ ]:
#| hide
# a runtime that does register a prober gets asked instead
register_runtime(replace(RUNTIMES['llama'],
                         caps=lambda model, model_path=None: Caps(('text', 'image'), source='runtime')))
test_eq(model_caps('qwen3-4b.gguf').source, 'runtime')
register_runtime(replace(RUNTIMES['llama'], caps=lambda *a, **kw: 1/0))   # a broken prober
test_eq(model_caps('qwen3-4b.gguf').source, 'default')                    # must not break a chat
register_runtime(replace(RUNTIMES['llama'], caps=None))

## Options

One name per thing, in three groups: what the conversation is, how the tool loop behaves, and how the model generates. Every field defaults to `None` or to a value meaning "unset", so `backend_kw` can tell an option the caller chose from one they never mentioned.

`extra` is the escape hatch for a knob only one backend has. It is passed through untouched, which keeps a rare option possible without adding a field nobody else uses.

In [ ]:
#| export
DFLT_FINAL_PROMPT = ("You've reached the tool-call budget for this turn. Stop calling tools and "
                     "answer with what you already have.")

#: Options that can also be set for a single turn, rather than for the whole chat.
GEN_OPTS = ('max_output_tokens', 'temp', 'top_k', 'top_p', 'seed', 'think', 'effort', 'tool_mode')

@dataclass(frozen=True)
class ChatOpts:
    "Everything a chat can be configured with, under names every runtime shares."
    # the conversation
    sp: str = ''                       # system prompt
    tools: tuple = ()                  # callables, or spec dicts for provider-side tools
    messages: tuple = ()               # history to start from
    # the tool loop
    approve: object = None             # `f(tool_call) -> bool`; None runs every call
    max_steps: int = 10                # tool-call budget for one turn
    tool_max_len: int = 0              # truncate a tool result past this many characters
    parallel_tools: bool = False
    max_parallel_tools: int = 0
    final_prompt: str = DFLT_FINAL_PROMPT   # sent once the budget runs out
    cbs: tuple = ()
    default_cbs: bool = True
    # generation
    ctx: int = 0                       # context window to ask the backend for
    max_output_tokens: int = 0
    temp: float = None
    top_k: int = None
    top_p: float = None
    seed: int = None
    think: bool = None                 # False asks a reasoning model not to deliberate
    effort: str = ''                   # 'low'|'medium'|'high'|... for models that grade it
    tool_mode: str = ''                # 'native' sends schemas on the wire, 'tags' in the prompt
    # credentials
    api_key: str = ''
    api_key_env: str = ''              # ...or the variable holding it, for config that must not
    base_url: str = ''                 # carry secrets
    # anything the portable names do not cover
    extra: dict = field(default_factory=dict)

    @classmethod
    def create(cls, opts=None, **kw):
        "A `ChatOpts` from an existing one plus overrides, or from loose keywords alone."
        if opts is None: opts = cls()
        elif isinstance(opts, dict): opts = cls(**opts)
        if not kw: return opts
        known = {f.name for f in fields(cls)}
        extra = {**opts.extra, **{k: v for k, v in kw.items() if k not in known}}
        return replace(opts, **{k: v for k, v in kw.items() if k in known}, extra=extra)

    @property
    def key(self):
        "The API key: the explicit one, else whatever `api_key_env` names. Never stored on the object."
        import os
        return self.api_key or (os.environ.get(self.api_key_env) if self.api_key_env else None) or None

    def set(self):
        "The options actually chosen, by name. Defaults are absent, so a backend can tell them apart."
        d = {}
        for f in fields(self):
            if f.name == 'extra': continue
            v = getattr(self, f.name)
            if v != f.default and not (f.default is None and v is None): d[f.name] = v
        return d

In [ ]:
o = ChatOpts.create(sp='Be brief.', temp=0.2)
test_eq((o.sp, o.temp, o.max_steps), ('Be brief.', 0.2, 10))
test_eq(o.set(), {'sp': 'Be brief.', 'temp': 0.2})   # only what was chosen

In [ ]:
# loose keywords with no field of their own land in `extra`, rather than raising
o = ChatOpts.create(temp=0.2, n_gpu_layers=99)
test_eq(o.extra, {'n_gpu_layers': 99})
test_eq(ChatOpts.create(o, temp=0.9).temp, 0.9)      # override an existing one
test_eq(ChatOpts.create(o, seed=1).extra, {'n_gpu_layers': 99})   # ...keeping the rest

In [ ]:
import os
os.environ['DEMO_KEY'] = 'sk-secret'
test_eq(ChatOpts.create(api_key_env='DEMO_KEY').key, 'sk-secret')
test_eq(ChatOpts.create(api_key='explicit', api_key_env='DEMO_KEY').key, 'explicit')
test_eq(ChatOpts.create(api_key_env='NOT_SET_ANYWHERE').key, None)
test_eq(ChatOpts().key, None)
del os.environ['DEMO_KEY']

In [ ]:
test_eq(ChatOpts.create({'sp': 'from a dict'}).sp, 'from a dict')
test_eq(ChatOpts.create(), ChatOpts())
test_eq(ChatOpts().set(), {})

## Translating for a backend

A backend declares two class attributes and nothing else: `_opt_map` renames the options it spells differently, and `_opt_skip` names the ones it cannot honour at all.

A dotted target nests, which is the whole of what litert needs: `{'ctx': 'eng_kw.max_num_tokens'}`.

Skipping is not silent. An option the caller explicitly set and the backend cannot honour produces a warning, because the alternative is a `temp=0` that quietly did nothing.

In [ ]:
#| export
def set_dotted(d, path, v):
    "Set `d['a']['b'] = v` for a path of `'a.b'`, creating the intermediate dicts."
    ks, cur = path.split('.'), d
    for k in ks[:-1]: cur = cur.setdefault(k, {})
    cur[ks[-1]] = v
    return d

def backend_kw(opts, opt_map=None, skip=(), warn=True):
    "`opts` as one backend's own keyword arguments, renamed by `opt_map` and minus `skip`."
    out, chosen = {}, opts.set()
    if (env := chosen.pop('api_key_env', None)) and 'api_key' not in chosen:
        import os
        if (v := os.environ.get(env)): chosen['api_key'] = v
    dropped = [k for k in chosen if k in skip]
    if warn and dropped:
        warnings.warn(f'ignored by this backend: {", ".join(sorted(dropped))}', stacklevel=2)
    for k, v in chosen.items():
        if k in skip: continue
        set_dotted(out, (opt_map or {}).get(k, k), v)
    return {**out, **opts.extra}

In [ ]:
test_eq(set_dotted({}, 'eng_kw.max_num_tokens', 8192), {'eng_kw': {'max_num_tokens': 8192}})
test_eq(set_dotted({'eng_kw': {'a': 1}}, 'eng_kw.b', 2), {'eng_kw': {'a': 1, 'b': 2}})
test_eq(set_dotted({}, 'plain', 1), {'plain': 1})

In [ ]:
o = ChatOpts.create(ctx=8192, temp=0.2, sp='hi')
test_eq(backend_kw(o, {'ctx': 'n_ctx'}), {'n_ctx': 8192, 'temp': 0.2, 'sp': 'hi'})
test_eq(backend_kw(o, {'ctx': 'eng_kw.max_num_tokens'}),
        {'eng_kw': {'max_num_tokens': 8192}, 'temp': 0.2, 'sp': 'hi'})

In [ ]:
# what a backend cannot do is dropped, and said out loud
import warnings as _w
with _w.catch_warnings(record=True) as caught:
    _w.simplefilter('always')
    test_eq(backend_kw(ChatOpts.create(temp=0.2, effort='high'), skip=('effort',)), {'temp': 0.2})
test_eq(len(caught), 1)
assert 'effort' in str(caught[0].message)

In [ ]:
# an option the caller never set is not "ignored" -- there was nothing to ignore
with _w.catch_warnings(record=True) as caught:
    _w.simplefilter('always')
    test_eq(backend_kw(ChatOpts.create(temp=0.2), skip=('effort',)), {'temp': 0.2})
test_eq(len(caught), 0)

In [ ]:
import os
os.environ['DEMO_KEY'] = 'sk-secret'
test_eq(backend_kw(ChatOpts.create(api_key_env='DEMO_KEY')), {'api_key': 'sk-secret'})
test_eq(backend_kw(ChatOpts.create(api_key_env='DEMO_KEY', api_key='explicit')),
        {'api_key': 'explicit'})            # an explicit key wins over the variable
test_eq(backend_kw(ChatOpts.create(api_key_env='NOT_SET_ANYWHERE')), {})
del os.environ['DEMO_KEY']

In [ ]:
# `extra` goes through last and untouched, so a backend-only knob always arrives
test_eq(backend_kw(ChatOpts.create(temp=0.2, n_gpu_layers=99), {'temp': 'temperature'}),
        {'temperature': 0.2, 'n_gpu_layers': 99})

## One turn at a time

Generation settings can change per turn without rebuilding the chat: which model thinks harder for a hard question, which one runs cold for an extraction. `turn_kw` picks out the settings that make sense for a single call and leaves the rest alone.

In [ ]:
#| export
def turn_kw(opts=None, **kw):
    "Generation settings for one turn, from a `ChatOpts` or loose keywords. Loop and conversation settings are ignored."
    o = ChatOpts.create(opts, **kw)
    chosen = o.set()
    return {**{k: v for k, v in chosen.items() if k in GEN_OPTS}, **o.extra}

In [ ]:
test_eq(turn_kw(temp=0.9, effort='high'), {'temp': 0.9, 'effort': 'high'})
test_eq(turn_kw(temp=0.9, sp='ignored here'), {'temp': 0.9})   # not a per-turn thing
test_eq(turn_kw(), {})
test_eq(turn_kw(ChatOpts(temp=0.1), temp=0.9), {'temp': 0.9})

In [ ]:
#| hide
RUNTIMES.clear()      # leave the registry as we found it: a real backend fills it

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()